In [4]:
!pip install timm -q

import os
import torch
import timm
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import f1_score, classification_report

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = "/kaggle/input/datasets/veeraiahkondra/vkt1234/Final_Data_CLAHE"
print("Using device:", device)

Using device: cuda


In [6]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.2,0.2),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

In [7]:
train_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=train_transform)
val_dataset   = datasets.ImageFolder(os.path.join(DATA_DIR, "val"), transform=val_transform)
test_dataset  = datasets.ImageFolder(os.path.join(DATA_DIR, "test"), transform=val_transform)

class_names = train_dataset.classes
print("Classes:", class_names)

# 🔥 Handle imbalance
targets = [label for _, label in train_dataset]
class_counts = np.bincount(targets)

class_weights = 1. / class_counts
sample_weights = [class_weights[t] for t in targets]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

Classes: ['Covid-19', 'Normal', 'Pneumonia-Bacterial', 'Pneumonia-Viral']


In [8]:
train_loader = DataLoader(train_dataset, batch_size=16, sampler=sampler)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [9]:
model_eff = timm.create_model('tf_efficientnet_b4', pretrained=True, num_classes=4).to(device)
model_inc = timm.create_model('inception_v3', pretrained=True, num_classes=4).to(device)

model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/95.5M [00:00<?, ?B/s]

In [10]:

criterion = nn.CrossEntropyLoss(label_smoothing=0.0)

optimizer_eff = optim.AdamW(model_eff.parameters(), lr=3e-4)
optimizer_inc = optim.AdamW(model_inc.parameters(), lr=3e-4)

scheduler_eff = optim.lr_scheduler.CosineAnnealingLR(optimizer_eff, T_max=30)
scheduler_inc = optim.lr_scheduler.CosineAnnealingLR(optimizer_inc, T_max=30)

In [11]:
def train_effnet(epochs=30):
    best_f1 = 0

    for epoch in range(epochs):
        model_eff.train()

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer_eff.zero_grad()
            outputs = model_eff(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer_eff.step()

        scheduler_eff.step()

        # VALIDATION
        model_eff.eval()
        preds, gts = [], []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)

                out = model_eff(images)
                pred = torch.argmax(out, dim=1)

                preds.extend(pred.cpu().numpy())
                gts.extend(labels.numpy())

        f1 = f1_score(gts, preds, average='macro')
        print(f"EffNet Epoch {epoch+1} | F1: {f1:.4f}")

        if f1 > best_f1:
            best_f1 = f1
            torch.save(model_eff.state_dict(), "best_eff.pth")

    print("✅ EfficientNet Done")

train_effnet()

EffNet Epoch 1 | F1: 0.8123
EffNet Epoch 2 | F1: 0.8563
EffNet Epoch 3 | F1: 0.8740
EffNet Epoch 4 | F1: 0.9036
EffNet Epoch 5 | F1: 0.8948
EffNet Epoch 6 | F1: 0.8931
EffNet Epoch 7 | F1: 0.8778
EffNet Epoch 8 | F1: 0.8901
EffNet Epoch 9 | F1: 0.9087
EffNet Epoch 10 | F1: 0.9259
EffNet Epoch 11 | F1: 0.9216
EffNet Epoch 12 | F1: 0.9270
EffNet Epoch 13 | F1: 0.9239
EffNet Epoch 14 | F1: 0.9286
EffNet Epoch 15 | F1: 0.9389
EffNet Epoch 16 | F1: 0.9301
EffNet Epoch 17 | F1: 0.9222
EffNet Epoch 18 | F1: 0.9296
EffNet Epoch 19 | F1: 0.9299
EffNet Epoch 20 | F1: 0.9312
EffNet Epoch 21 | F1: 0.9435
EffNet Epoch 22 | F1: 0.9384
EffNet Epoch 23 | F1: 0.9445
EffNet Epoch 24 | F1: 0.9510
EffNet Epoch 25 | F1: 0.9496
EffNet Epoch 26 | F1: 0.9492
EffNet Epoch 27 | F1: 0.9467
EffNet Epoch 28 | F1: 0.9486
EffNet Epoch 29 | F1: 0.9485
EffNet Epoch 30 | F1: 0.9460
✅ EfficientNet Done


In [12]:
def train_inception(epochs=30):
    best_f1 = 0

    for epoch in range(epochs):
        model_inc.train()

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # 🔥 resize ONLY here
            images_299 = F.interpolate(images, size=(299,299), mode='bilinear')

            optimizer_inc.zero_grad()
            outputs = model_inc(images_299)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer_inc.step()

        scheduler_inc.step()

        # VALIDATION
        model_inc.eval()
        preds, gts = [], []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                images_299 = F.interpolate(images, size=(299,299), mode='bilinear')

                out = model_inc(images_299)
                pred = torch.argmax(out, dim=1)

                preds.extend(pred.cpu().numpy())
                gts.extend(labels.numpy())

        f1 = f1_score(gts, preds, average='macro')
        print(f"Inception Epoch {epoch+1} | F1: {f1:.4f}")

        if f1 > best_f1:
            best_f1 = f1
            torch.save(model_inc.state_dict(), "best_inc.pth")

    print("✅ Inception Done")

train_inception()

Inception Epoch 1 | F1: 0.8693
Inception Epoch 2 | F1: 0.8592
Inception Epoch 3 | F1: 0.8361
Inception Epoch 4 | F1: 0.8773
Inception Epoch 5 | F1: 0.8419
Inception Epoch 6 | F1: 0.8728
Inception Epoch 7 | F1: 0.8831
Inception Epoch 8 | F1: 0.8586
Inception Epoch 9 | F1: 0.8763
Inception Epoch 10 | F1: 0.8879
Inception Epoch 11 | F1: 0.9084
Inception Epoch 12 | F1: 0.9062
Inception Epoch 13 | F1: 0.9028
Inception Epoch 14 | F1: 0.9166
Inception Epoch 15 | F1: 0.8660
Inception Epoch 16 | F1: 0.8937
Inception Epoch 17 | F1: 0.9242
Inception Epoch 18 | F1: 0.9331
Inception Epoch 19 | F1: 0.9317
Inception Epoch 20 | F1: 0.9333
Inception Epoch 21 | F1: 0.9418
Inception Epoch 22 | F1: 0.9400
Inception Epoch 23 | F1: 0.9423
Inception Epoch 24 | F1: 0.9430
Inception Epoch 25 | F1: 0.9379
Inception Epoch 26 | F1: 0.9441
Inception Epoch 27 | F1: 0.9434
Inception Epoch 28 | F1: 0.9436
Inception Epoch 29 | F1: 0.9448
Inception Epoch 30 | F1: 0.9423
✅ Inception Done


In [13]:
model_eff.load_state_dict(torch.load("best_eff.pth", map_location=device))
model_inc.load_state_dict(torch.load("best_inc.pth", map_location=device))

model_eff.eval()
model_inc.eval()

InceptionV3(
  (Conv2d_1a_3x3): ConvNormAct(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNormAct2d(
      32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): ReLU(inplace=True)
    )
  )
  (Conv2d_2a_3x3): ConvNormAct(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNormAct2d(
      32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): ReLU(inplace=True)
    )
  )
  (Conv2d_2b_3x3): ConvNormAct(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNormAct2d(
      64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): ReLU(inplace=True)
    )
  )
  (Pool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Conv2d_3b_1x1): ConvNormAct(
    (conv): Conv2d(64, 80, kernel_size

In [14]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
import torch
import numpy as np

# ==============================
# TTA TRANSFORMS
# ==============================
def tta_transforms(images):

    # 1️⃣ Original
    t1 = images

    # 2️⃣ Horizontal flip
    t2 = torch.flip(images, dims=[3])

    # 3️⃣ Slight brightness increase
    t3 = torch.clamp(images * 1.03, 0, 1)

    # 4️⃣ Slight brightness decrease
    t4 = torch.clamp(images * 0.97, 0, 1)

    return [t1, t2, t3, t4]


# ==============================
# ENSEMBLE WEIGHTS
# ==============================
w_eff = 0.6
w_inc = 0.4

# TTA boost weights
boosts = [1, 1, 1.6, 1.6]

# ==============================
# EVALUATION
# ==============================
model_eff.eval()
model_inc.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Apply TTA
        tta_imgs = tta_transforms(images)

        # Store predictions
        final_probs = None

        # ==========================
        # TTA LOOP
        # ==========================
        for idx, tta_img in enumerate(tta_imgs):

            # EfficientNet prediction
            out_eff = model_eff(tta_img)

            # Inception prediction
            out_inc = model_inc(tta_img)

            # If inception returns auxiliary outputs
            if isinstance(out_inc, tuple):
                out_inc = out_inc[0]

            # Softmax probabilities
            prob_eff = torch.softmax(out_eff, dim=1)
            prob_inc = torch.softmax(out_inc, dim=1)

            # Weighted ensemble
            ensemble_prob = (
                w_eff * prob_eff +
                w_inc * prob_inc
            )

            # Apply TTA boost
            ensemble_prob = boosts[idx] * ensemble_prob

            # Accumulate
            if final_probs is None:
                final_probs = ensemble_prob
            else:
                final_probs += ensemble_prob

        # Average TTA predictions
        final_probs = final_probs / sum(boosts)

        # Final prediction
        preds = torch.argmax(final_probs, dim=1)

        # Store
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


# ==============================
# METRICS
# ==============================
accuracy = accuracy_score(all_labels, all_preds)
macro_f1 = f1_score(all_labels, all_preds, average='macro')

print(f"\n✅ Accuracy  : {accuracy:.4f}")
print(f"✅ Macro F1  : {macro_f1:.4f}")

# Optional detailed report
print("\n📊 Classification Report:\n")
print(classification_report(all_labels, all_preds))


✅ Accuracy  : 0.9457
✅ Macro F1  : 0.9458

📊 Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       405
           1       0.99      0.99      0.99       405
           2       0.91      0.89      0.90       405
           3       0.88      0.91      0.90       405

    accuracy                           0.95      1620
   macro avg       0.95      0.95      0.95      1620
weighted avg       0.95      0.95      0.95      1620



In [15]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import random

from sklearn.metrics import (
    f1_score,
    accuracy_score
)

In [16]:
def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False

In [17]:
def ensemble_predict(
    images,
    w1,
    w2,
    boost
):

    img224 = images

    img299 = F.interpolate(
        images,
        size=(299,299),
        mode='bilinear',
        align_corners=False
    )

    logits_eff = model_eff(img224)

    logits_inc = model_inc(img299)

    p_eff = torch.softmax(
        logits_eff,
        dim=1
    )

    p_inc = torch.softmax(
        logits_inc,
        dim=1
    )

    pfusion = (
        w1 * p_eff +
        w2 * p_inc
    )

    boost = boost.unsqueeze(0)

    pboost = pfusion * boost

    pfinal = pboost / pboost.sum(
        dim=1,
        keepdim=True
    )

    return pfinal

In [18]:
import random

def tta_predict(
    images,
    w1,
    w2,
    boost
):

    outputs = []

    for _ in range(5):

        aug = images.clone()

        # ==========================================
        # RANDOM HORIZONTAL FLIP
        # ==========================================

        if random.random() > 0.5:

            aug = torch.flip(
                aug,
                dims=[3]
            )

        # ==========================================
        # RANDOM BRIGHTNESS
        # ==========================================

        brightness = np.random.uniform(
            0.95,
            1.05
        )

        aug = torch.clamp(
            aug * brightness,
            0,
            1
        )

        # ==========================================
        # RANDOM GAUSSIAN NOISE
        # ==========================================

        noise = torch.randn_like(aug) * 0.01

        aug = torch.clamp(
            aug + noise,
            0,
            1
        )

        # ==========================================
        # PREDICTION
        # ==========================================

        out = ensemble_predict(
            aug,
            w1,
            w2,
            boost
        )

        outputs.append(out)

    outputs = torch.stack(outputs)

    return outputs.mean(dim=0)

In [19]:
def evaluate_model(
    loader,
    w1,
    w2,
    boost,
    use_tta=True
):

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            labels = labels.to(device)

            if use_tta:

                outputs = tta_predict(
                    images,
                    w1,
                    w2,
                    boost
                )

            else:

                outputs = ensemble_predict(
                    images,
                    w1,
                    w2,
                    boost
                )

            preds = torch.argmax(
                outputs,
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average='macro'
    )

    accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    return macro_f1, accuracy

In [20]:
SEEDS = [42,52,62,72,82]

boost_tensor = torch.tensor(
    [1.0,1.0,1.6,1.6],
    device=device
)

scores = []

for seed in SEEDS:

    set_seed(seed)

    f1, acc = evaluate_model(
        val_loader,
        w1=0.6,
        w2=0.4,
        boost=boost_tensor,
        use_tta=True
    )

    scores.append(f1)

mean_f1 = np.mean(scores)

std_f1 = np.std(scores)

print("WITHOUT LABEL SMOOTHING")

print(
    f"Val Macro F1 Mean ± SD: "
    f"{mean_f1:.4f} ± {std_f1:.4f}"
)

WITHOUT LABEL SMOOTHING
Val Macro F1 Mean ± SD: 0.9402 ± 0.0014


In [21]:
test_f1, test_acc = evaluate_model(
    test_loader,
    w1=0.6,
    w2=0.4,
    boost=boost_tensor,
    use_tta=True
)

print("WITHOUT LABEL SMOOTHING")

print(
    "Test Macro F1:",
    round(test_f1,4)
)

print(
    "Test Accuracy:",
    round(test_acc,4)
)

WITHOUT LABEL SMOOTHING
Test Macro F1: 0.9312
Test Accuracy: 0.9315
